# 🔬 Purple Agent — Baseline vs Current Agent Backtest

**Amaç:** Yeni `LangGraphAgent` mimarisinin eski sade ReAct'a göre performansını ölçmek.

| | Mimari |
|---|---|
| **Baseline** | Sade ReAct — `create_agent`, middleware yok, düz system prompt |
| **Candidate** | Yeni `LangGraphAgent` — `StructuredTool` + Pydantic schema + middleware stack + focus injection |

**Dataset:** `purple-agent-benchmark-v1`  
**Model:** `gpt-4o-mini` (her ikisi için sabit)  
**Metrikler:** `tool_usage_recall`, `task_success`


In [3]:
import os
import sys
import asyncio
import uuid
from dotenv import load_dotenv

# langgraph_agent module-level asyncio.run() Jupyter event loop'u öldürüyor
# Bu satır o bloğu devre dışı bırakır
os.environ["SKIP_AGENT_INIT"] = "true"

_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), "../../../"))
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

load_dotenv()

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "purple-agent-backtest")

MCP_ENDPOINT  = os.getenv("MCP_ENDPOINT", "http://localhost:8091").removesuffix("/mcp")
DATASET_NAME  = "purple-agent-benchmark-v1"
MODEL_NAME    = "local-qwen"   # LocalModel kullanılıyor

print(f"MCP Endpoint : {MCP_ENDPOINT}")
print(f"Dataset      : {DATASET_NAME}")
print(f"Model        : {MODEL_NAME}")
print(f"LangSmith    : {os.getenv('LANGSMITH_PROJECT')}")


MCP Endpoint : http://localhost:8091
Dataset      : purple-agent-benchmark-v1
Model        : local-qwen
LangSmith    : langchain-app


## 1️⃣ Agent Kurulumu

**Baseline:** Düz `create_agent` ReAct döngüsü — middleware yok, en saf ölçüm.  
**Candidate:** `LangGraphAgent` — `StructuredTool` + Pydantic schema + `_log_model_call` + `ToolRetry` + `_handle_tool_errors` middleware stack.


In [ ]:
import httpx
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ToolRetryMiddleware,
    wrap_tool_call,
    before_model,
)
from langchain_core.messages import ToolMessage

from src.purple_agent.langgraph_agent import (
    LangGraphAgent,
    MCPToolLoader,
    _log_model_call,
    _handle_tool_errors,
)

from custom_qwen import LocalModel

# ── Shared: MCP tool loader ──────────────────────────────────────────────────
_tool_loader = MCPToolLoader(MCP_ENDPOINT)

# ── Baseline: sade ReAct, middleware YOK ────────────────────────────────────
_BASELINE_SYSTEM_PROMPT = """\
You are a capable AI assistant. You have access to tools.
Use them to complete the task accurately and efficiently.
Always provide ALL required arguments when calling a tool.
On error, adjust arguments and retry.
"""

def build_baseline_graph(tools):
    """Düz create_agent — middleware yok, en sade ReAct."""
    return create_agent(
        model=LocalModel(),
        tools=tools,
        system_prompt=_BASELINE_SYSTEM_PROMPT,
    )


# ── Candidate: LangGraphAgent (production) ──────────────────────────────────
_candidate_agent: LangGraphAgent | None = None

async def _build_candidate_agent() -> LangGraphAgent:
    """LangGraphAgent'ı başlatır; StructuredTool + middleware stack."""
    agent = LangGraphAgent(
        mcp_endpoint=MCP_ENDPOINT,
        model=LocalModel(),
    )
    await agent.initialize()
    return agent


# ── Lazy-initialized state ───────────────────────────────────────────────────
_shared_tools = None            # MCP tools (baseline için)
_baseline_graph = None          # Baseline sade ReAct graph
_candidate_agent_cache = None   # Candidate LangGraphAgent instance

print("✅ Agent kurulumu tamamlandı")


✅ Agent kurulumu tamamlandı (modül reload edildi)


## 2️⃣ Target Functions & Evaluators

Her iki ajan için ayrı `target` fonksiyonu. Evaluator'lar paylaşımlı.

In [ ]:
from langchain_core.messages import HumanMessage
from langsmith import Client
from langsmith.schemas import Example, Run

# ── Shared tools loader (baseline için) ─────────────────────────────────────
async def _ensure_tools() -> list:
    global _shared_tools
    if _shared_tools is None:
        _shared_tools = await _tool_loader.load_tools()
        print(f"✅ MCP tools yüklendi ({len(_shared_tools)} tool)")
    return _shared_tools


# ── Baseline: sade ReAct graph cache ────────────────────────────────────────
async def _ensure_baseline():
    global _baseline_graph
    if _baseline_graph is None:
        tools = await _ensure_tools()
        _baseline_graph = build_baseline_graph(tools)
        print("✅ Baseline graph hazır")
    return _baseline_graph


# ── Candidate: LangGraphAgent cache ─────────────────────────────────────────
async def _ensure_candidate() -> LangGraphAgent:
    global _candidate_agent_cache
    if _candidate_agent_cache is None:
        _candidate_agent_cache = await _build_candidate_agent()
        print("✅ Candidate LangGraphAgent hazır")
    return _candidate_agent_cache


# ── Helpers ──────────────────────────────────────────────────────────────────
def _extract_question(inputs: dict) -> str:
    for key in ("text", "question", "input"):
        if inputs.get(key):
            return inputs[key]
    return next((v for v in inputs.values() if isinstance(v, str)), "")


# ── Target Functions ─────────────────────────────────────────────────────────
async def target_baseline(inputs: dict) -> dict:
    graph = await _ensure_baseline()
    question = _extract_question(inputs)
    if not question:
        return {"error": "No question found", "messages": []}
    try:
        config = {"configurable": {"thread_id": str(uuid.uuid4())}}
        return await graph.ainvoke(
            {"messages": [HumanMessage(content=question)]},
            config=config,
        )
    except Exception as e:
        return {"error": str(e), "messages": []}


async def target_candidate(inputs: dict) -> dict:
    agent = await _ensure_candidate()
    question = _extract_question(inputs)
    if not question:
        return {"error": "No question found", "messages": []}
    try:
        config = {"configurable": {"thread_id": str(uuid.uuid4())}}
        return await agent.graph.ainvoke(
            {"messages": [HumanMessage(content=question)]},
            config=config,
        )
    except Exception as e:
        return {"error": str(e), "messages": []}


# ── Evaluators ───────────────────────────────────────────────────────────────
def tool_usage_recall(run: Run, example: Example) -> dict:
    """Beklenen toolların kaçının çağrıldığını ölçer (recall)."""
    expected: list = (example.outputs or {}).get("expected_tools", [])
    if not expected:
        return {"key": "tool_usage_recall", "score": None}

    messages = (run.outputs or {}).get("messages", [])
    called = set()
    for msg in messages:
        tool_calls = (
            getattr(msg, "tool_calls", None)
            or (msg.get("tool_calls") if isinstance(msg, dict) else None)
            or (msg.get("kwargs", {}).get("tool_calls") if isinstance(msg, dict) else None)
            or []
        )
        for tc in tool_calls:
            name = tc.get("name") if isinstance(tc, dict) else getattr(tc, "name", None)
            if name:
                called.add(name)

    missing = [t for t in expected if t not in called]
    score   = (len(expected) - len(missing)) / len(expected)
    return {
        "key": "tool_usage_recall",
        "score": round(score, 3),
        "comment": f"expected={expected} | called={sorted(called)} | missing={missing}",
    }


def task_success(run: Run, example: Example) -> dict:
    """Agent hata döndürmeden tamamladıysa 1, aksi hâlde 0."""
    outputs = run.outputs or {}
    has_error = bool(outputs.get("error"))
    messages  = outputs.get("messages", [])
    has_answer = any(
        getattr(m, "content", None) or (isinstance(m, dict) and m.get("content"))
        for m in messages[-2:] if not getattr(m, "tool_calls", None)
    )
    score = 0 if has_error else (1 if has_answer else 0)
    return {"key": "task_success", "score": score}


print("✅ Target fonksiyonları ve evaluator'lar hazır")


✅ Target fonksiyonları ve evaluator'lar hazır
   'task' key kullanılıyor (input yerine — create_agent çakışması önlendi)


## 3️⃣ Baseline Experiment — Sade ReAct (middleware yok)

`create_agent` ile tek döngü. Middleware stack yok, system prompt sabit, en saf ölçüm.


In [ ]:
from langsmith import aevaluate

_run_id = str(uuid.uuid4())[:6]
BASELINE_PREFIX = f"baseline-react-{MODEL_NAME}-{_run_id}"

print(f"🚀 Baseline başlıyor: {BASELINE_PREFIX}")
print(f"   Mimari : Sade ReAct — middleware YOK, en saf ölçüm\n")

baseline_results = await aevaluate(
    target_baseline,
    data=DATASET_NAME,
    evaluators=[tool_usage_recall, task_success],
    experiment_prefix=BASELINE_PREFIX,
    max_concurrency=1,
    description="Baseline: Düz create_agent ReAct — middleware yok",
    metadata={"model": MODEL_NAME, "architecture": "react-no-middleware", "variant": "baseline"},
)

print(f"\n✅ Baseline tamamlandı → {baseline_results.experiment_name}")


## 4️⃣ Candidate Experiment — LangGraphAgent (production)

`LangGraphAgent` — `StructuredTool` + Pydantic schema + `_log_model_call` (focus injection) + `ToolRetry` + `_handle_tool_errors`.


In [ ]:
from langsmith import aevaluate

CANDIDATE_PREFIX = f"candidate-langgraph-{MODEL_NAME}-"

print(f"🚀 Candidate başlıyor: {CANDIDATE_PREFIX}")
print(f"   Mimari : LangGraphAgent — StructuredTool + middleware stack\n")

candidate_results = await aevaluate(
    target_candidate,
    data=DATASET_NAME,
    evaluators=[tool_usage_recall, task_success],
    experiment_prefix=CANDIDATE_PREFIX,
    max_concurrency=1,
    description="Candidate: LangGraphAgent — StructuredTool + focus injection + ToolRetry",
    metadata={"model": MODEL_NAME, "architecture": "langgraph-agent", "variant": "candidate"},
)

print(f"\n✅ Candidate tamamlandı → {candidate_results.experiment_name}")


🚀 Candidate başlıyor: candidate-plan-execute-local-qwen-
   Mimari : Plan-Execute (planner node → executor node)

View the evaluation results for experiment: 'candidate-plan-execute-local-qwen--9bfd0c48' at:
https://smith.langchain.com/o/378a72ed-098a-4298-9931-7e1696a11886/datasets/b45ac299-b3c4-415e-8e09-c508d5d02fa4/compare?selectedSessions=e146e28a-6f8a-45f7-8bb4-49149437d40b




0it [00:00, ?it/s]

[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-Instruct-2507:
  tools count: 0
  tool_choice: not set
[LocalModel] Response received:
  content: ["search for 'Quarterly Review' document in Google Drive", "getGoogleDocContent to retrieve the cont...
  tool_calls: []
  finish_reason: stop
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-Instruct-2507:
  tools count: 76
  tool_choice: auto
[LocalModel] Response received:
  content: (none)...
  tool_calls: [{'id': 'chatcmpl-tool-df5236f75cbd47debaf013bd0b0ab476', 'type': 'function', 'function': {'name': 'search', 'arguments': '{"query": "name=\'Quarterly Review\' and mimeType=\'application/vnd.google-apps.document\'", "pageSize": 50, "pageToken": null}'}}]
  finish_reason: tool_calls
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-Instruct-2507:
  tools count: 76
  tool_choice: auto
[LocalModel] Response received:
  content: (none)...
  tool_calls: [{'id': 'chatcmpl-tool-baceb8

1it [00:32, 32.28s/it]

[LocalModel] Response received:
  content: I'm encountering a technical limitation with executing any tool calls due to an asynchronous executi...
  tool_calls: []
  finish_reason: stop
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-Instruct-2507:
  tools count: 0
  tool_choice: not set
[LocalModel] Response received:
  content: ["google_search for YouTube video 'MKBHD iPhone Review'", "get_video_info from the search result to ...
  tool_calls: []
  finish_reason: stop
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-Instruct-2507:
  tools count: 76
  tool_choice: auto
[LocalModel] Response received:
  content: (none)...
  tool_calls: [{'id': 'chatcmpl-tool-dcdda0888b4345e99800b643094c1de8', 'type': 'function', 'function': {'name': 'google_search', 'arguments': '{"q": "MKBHD iPhone Review", "num": 10, "tbs": "", "page": 1, "autocorrect": true}'}}]
  finish_reason: tool_calls
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-I

...

[LocalModel] Response received:
  content: I'm unable to interact with the Notion API due to technical limitations. However, I can provide the ...
  tool_calls: []
  finish_reason: stop
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-Instruct-2507:
  tools count: 0
  tool_choice: not set
[LocalModel] Response received:
  content: ["google_search for upcoming AI conferences in San Francisco Q3 2024", "scrape top results for confe...
  tool_calls: []
  finish_reason: stop
[LocalModel] Request to /gpfs/scratch/ehpc142/models/Qwen3-235B-A22B-Instruct-2507:
  tools count: 76
  tool_choice: auto
[LocalModel] Response received:
  content: (none)...
  tool_calls: [{'id': 'chatcmpl-tool-4510b4310c5a4077a6e6cd58f15e055a', 'type': 'function', 'function': {'name': 'google_search', 'arguments': '{"q": "upcoming AI conferences in San Francisco Q3 2024", "num": 10, "tbs": "", "page": 1, "autocorrect": true}'}}]
  finish_reason: tool_calls
[LocalModel] Request to /gpfs/scratch/ehp

20it [15:47, 46.37s/it]

[LocalModel] Response received:
  content: It seems that the `google_search` tool is currently unavailable due to a technical limitation in the...
  tool_calls: []
  finish_reason: stop


20it [15:48, 47.41s/it]


✅ Candidate tamamlandı → candidate-plan-execute-local-qwen--9bfd0c48


## 5️⃣ Sonuç Karşılaştırması

İki experiment'ın metriklerini yan yana karşılaştır.

In [ ]:
def _summarize(results, label: str) -> dict:
    rows = list(results)
    if not rows:
        return {"label": label, "n": 0}

    def _avg(key: str) -> float:
        scores = [
            r.get("feedback", {}).get(key)
            for r in rows
            if r.get("feedback", {}).get(key) is not None
        ]
        return round(sum(scores) / len(scores), 3) if scores else float("nan")

    return {
        "label":             label,
        "n":                 len(rows),
        "tool_usage_recall": _avg("tool_usage_recall"),
        "task_success":      _avg("task_success"),
    }


b = _summarize(baseline_results,  "Baseline  (ReAct, middleware yok)")
c = _summarize(candidate_results, "Candidate (LangGraphAgent)")

header = f"{'Metric':<25} {'Baseline':>14} {'Candidate':>14} {'Delta':>10}"
sep    = "─" * len(header)
print(sep)
print(header)
print(sep)

for m in ["tool_usage_recall", "task_success"]:
    bv = b.get(m, float("nan"))
    cv = c.get(m, float("nan"))
    try:
        delta  = f"{cv - bv:+.3f}"
        winner = "✅" if cv >= bv else "⚠️"
    except TypeError:
        delta, winner = "N/A", ""
    print(f"{m:<25} {bv:>14} {cv:>14} {delta:>10}  {winner}")

print(sep)
print(f"\n📌 Örnek sayısı : {b['n']}")
print(f"📊 Baseline     : {BASELINE_PREFIX}")
print(f"📊 Candidate    : {CANDIDATE_PREFIX}")
print(f"🔗 LangSmith    : https://smith.langchain.com")


NameError: name 'baseline_results' is not defined

Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /runs/multipart (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 8] nodename nor servname provided, or not known)"))'))
Content-Length: 535833
API Key: lsv2_********************************************7ftrace=019c7626-50b8-7861-9dea-506a55570cad,id=019c7640-5b2b-7851-9f36-8ae463cf885f; trace=019c7626-50b8-7861-9dea-506a55570cad,id=019c7640-5a90-75d1-8310-354ef0f0cb4d; trace=019c7626-50b8-7861-9dea-506a55570cad,id=019c7640-7cf8-7ee2-ab3b-a597d614cbe2; trace=019c7626-50b8-7861-9dea-506a55570cad,id=019c7640-7cf8-7ee2-ab3b-a597d614cbe2; trace=019c7626-50b8-7861-9dea-506a55570cad,id=

: 

: 

: 